In [1]:
# セットアップ: AutoAttack + DDPM防御評価
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import os
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

# PCamデータセットのパス（ddpm_fgsm と同じ）
DATA_DIR = '/mnt/data1/gotou/kaggle/pcam'
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train')
LABELS_CSV = os.path.join(DATA_DIR, 'train_labels.csv')

# データ変換
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PCamDataset(Dataset):
    def __init__(self, img_dir, labels_df, transform=None):
        self.img_dir = img_dir
        self.labels = labels_df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        img_id = self.labels.iloc[idx, 0]
        label = self.labels.iloc[idx, 1]
        img_path = os.path.join(self.img_dir, f"{img_id}.tif")
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

labels_df = pd.read_csv(LABELS_CSV)
_, val_df = train_test_split(labels_df, test_size=0.1, random_state=42, stratify=labels_df['label'])

val_dataset = PCamDataset(TRAIN_IMG_DIR, val_df, val_transform)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [3]:
# モデル読み込み（ResNet50分類器）
import torch.nn as nn
from torchvision import models

model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

ckpt_path = "/mnt/data1/gotou/kaggle/pcam/best_model_weights.pth"
state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print(f"✅ Loaded classifier from {ckpt_path}")


/home/gotou/miniconda3/envs/env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/gotou/miniconda3/envs/env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ Loaded classifier from /mnt/data1/gotou/kaggle/pcam/best_model_weights.pth


## AutoAttack の準備

AutoAttackライブラリのインストールが必要です:
```bash
pip install autoattack
```

AutoAttackは以下の4つの攻撃を自動的に実行します:
- **APGD-CE** (Auto-PGD with Cross-Entropy loss)
- **APGD-DLR** (Auto-PGD with DLR loss)  
- **FAB** (Fast Adaptive Boundary)
- **Square Attack**

バイナリ分類の場合、AutoAttackは自動的に適切な設定を使用します。

In [4]:
# AutoAttack のインポートと設定
from autoattack import AutoAttack

# AutoAttackの設定
# - norm: 'Linf' (L∞ノルム制約)
# - eps: 8/255 (摂動の最大値)
# - version: 'standard' (4つの攻撃すべて)
# - verbose: True (進捗表示)
epsilon = 8.0 / 255.0

# バイナリ分類用のラッパー
class BinaryClassifierWrapper(nn.Module):
    """
    AutoAttackはマルチクラス分類を想定しているため、
    バイナリ分類器をラップして2クラス確率を出力する
    """
    def __init__(self, binary_model):
        super().__init__()
        self.model = binary_model
    
    def forward(self, x):
        # binary_modelの出力: (batch_size, 1) のロジット
        logits = self.model(x)
        # シグモイドで確率に変換
        prob_positive = torch.sigmoid(logits)
        prob_negative = 1 - prob_positive
        # (batch_size, 2) の確率テンソルを返す
        return torch.cat([prob_negative, prob_positive], dim=1)

# ラッパーモデルを作成
wrapped_model = BinaryClassifierWrapper(model).to(device).eval()

print("AutoAttack設定完了")
print(f"Epsilon: {epsilon:.4f}")
print(f"Norm: L∞")
print(f"攻撃手法: APGD-CE, APGD-DLR, FAB, Square")

AutoAttack設定完了
Epsilon: 0.0314
Norm: L∞
攻撃手法: APGD-CE, APGD-DLR, FAB, Square


## DDPM モデルの定義と読み込み

拡散モデルによる浄化処理のためのモデル定義

In [5]:
# DDPM用のモデル定義（ddpm.pyと一致させる）
import math

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        device = t.device
        half = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half - 1)
        emb = torch.exp(torch.arange(half, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim=None):
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8 if out_ch >= 8 else 1, out_ch)
        self.norm2 = nn.GroupNorm(8 if out_ch >= 8 else 1, out_ch)
        if in_ch != out_ch:
            self.skip = nn.Conv2d(in_ch, out_ch, 1)
        else:
            self.skip = nn.Identity()
        if time_emb_dim is not None:
            self.time_mlp = nn.Sequential(
                nn.Linear(time_emb_dim, out_ch),
                nn.SiLU()
            )
        else:
            self.time_mlp = None
        self.act = nn.SiLU()
    def forward(self, x, t_emb=None):
        h = self.norm1(self.conv1(x))
        if self.time_mlp is not None and t_emb is not None:
            time_emb = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
            h = h + time_emb
        h = self.act(h)
        h = self.norm2(self.conv2(h))
        h = self.act(h)
        return h + self.skip(x)

class SimpleUNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, time_emb_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 2),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 2, time_emb_dim)
        )
        # down
        self.enc1 = ResidualBlock(in_ch, base_ch, time_emb_dim)
        self.down1 = nn.Conv2d(base_ch, base_ch*2, 4, stride=2, padding=1)
        self.enc2 = ResidualBlock(base_ch*2, base_ch*2, time_emb_dim)
        self.down2 = nn.Conv2d(base_ch*2, base_ch*4, 4, stride=2, padding=1)
        self.enc3 = ResidualBlock(base_ch*4, base_ch*4, time_emb_dim)
        self.down3 = nn.Conv2d(base_ch*4, base_ch*8, 4, stride=2, padding=1)
        self.enc4 = ResidualBlock(base_ch*8, base_ch*8, time_emb_dim)
        self.down4 = nn.Conv2d(base_ch*8, base_ch*8, 4, stride=2, padding=1)
        # bottleneck
        self.bot1 = ResidualBlock(base_ch*8, base_ch*8, time_emb_dim)
        self.bot2 = ResidualBlock(base_ch*8, base_ch*8, time_emb_dim)
        # up
        self.up4 = nn.ConvTranspose2d(base_ch*8, base_ch*8, 4, stride=2, padding=1)
        self.dec4 = ResidualBlock(base_ch*16, base_ch*8, time_emb_dim)
        self.up3 = nn.ConvTranspose2d(base_ch*8, base_ch*4, 4, stride=2, padding=1)
        self.dec3 = ResidualBlock(base_ch*8, base_ch*4, time_emb_dim)
        self.up2 = nn.ConvTranspose2d(base_ch*4, base_ch*2, 4, stride=2, padding=1)
        self.dec2 = ResidualBlock(base_ch*4, base_ch*2, time_emb_dim)
        self.up1 = nn.ConvTranspose2d(base_ch*2, base_ch, 4, stride=2, padding=1)
        self.dec1 = ResidualBlock(base_ch*2, base_ch, time_emb_dim)
        self.out_conv = nn.Sequential(
            nn.GroupNorm(8, base_ch),
            nn.SiLU(),
            nn.Conv2d(base_ch, in_ch, 3, padding=1)
        )
    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        e1 = self.enc1(x, t_emb)
        d1 = self.down1(e1)
        e2 = self.enc2(d1, t_emb)
        d2 = self.down2(e2)
        e3 = self.enc3(d2, t_emb)
        d3 = self.down3(e3)
        e4 = self.enc4(d3, t_emb)
        d4 = self.down4(e4)
        b1 = self.bot1(d4, t_emb)
        b2 = self.bot2(b1, t_emb)
        u4 = self.up4(b2)
        u4 = torch.cat([u4, e4], dim=1)
        u4 = self.dec4(u4, t_emb)
        u3 = self.up3(u4)
        u3 = torch.cat([u3, e3], dim=1)
        u3 = self.dec3(u3, t_emb)
        u2 = self.up2(u3)
        u2 = torch.cat([u2, e2], dim=1)
        u2 = self.dec2(u2, t_emb)
        u1 = self.up1(u2)
        u1 = torch.cat([u1, e1], dim=1)
        u1 = self.dec1(u1, t_emb)
        return self.out_conv(u1)

# DDPMモデルのロード（EMA対応版）
ddpm_model = SimpleUNet(in_ch=3, base_ch=64, time_emb_dim=256).to(device)
ddpm_checkpoint_path = "/mnt/data1/gotou/kaggle/path/ddpm_out/ddpm1_epoch10.pth"
ckpt = torch.load(ddpm_checkpoint_path, map_location=device)

# ddpm.py の保存形式に対応
if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
    try:
        # EMA があれば優先的に使用
        if 'ema_state_dict' in ckpt and isinstance(ckpt['ema_state_dict'], dict):
            ddpm_model.load_state_dict(ckpt['ema_state_dict'], strict=False)
            print("✅ EMA weights loaded")
        else:
            ddpm_model.load_state_dict(ckpt['model_state_dict'])
            print("✅ Model weights loaded (no EMA)")
    except Exception as e:
        print(f"⚠️ Error loading EMA, using model_state_dict: {e}")
        ddpm_model.load_state_dict(ckpt['model_state_dict'])
else:
    ddpm_model.load_state_dict(ckpt)
    print("✅ Direct state_dict loaded")

ddpm_model.eval()

print(f"DDPMモデル読み込み完了")
print(f"Checkpoint: {ddpm_checkpoint_path}")


✅ Model weights loaded (no EMA)
DDPMモデル読み込み完了
Checkpoint: /mnt/data1/gotou/kaggle/path/ddpm_out/ddpm1_epoch10.pth


In [6]:
# DDPM浄化用の関数定義
import numpy as np

# ベータスケジュールの設定
T = 1000
beta_start = 1e-4
beta_end = 0.02
betas = torch.linspace(beta_start, beta_end, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

# 正規化変換関数
def prepare_for_diffusion_from_norm(img_batch):
    """
    ImageNet正規化済み画像を拡散モデル用[-1,1]に変換
    """
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    
    # 逆正規化: [-1,1]区間へ
    img = img_batch * std + mean
    img = img.clamp(0, 1)
    img = img * 2.0 - 1.0
    return img

def prepare_from_diffusion_to_norm(img_batch):
    """
    拡散モデル出力[-1,1]をImageNet正規化に変換
    """
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    
    # [-1,1] -> [0,1]
    img = (img_batch + 1.0) / 2.0
    img = img.clamp(0, 1)
    # ImageNet正規化
    img = (img - mean) / std
    return img

def diffusion_purify(x_adv, model, t_start, T_purify, device):
    """
    拡散モデルによる浄化処理
    
    Args:
        x_adv: 敵対的サンプル（ImageNet正規化済み）
        model: DDPMモデル
        t_start: ノイズ付加の開始タイムステップ
        T_purify: 浄化ステップ数
        device: デバイス
    
    Returns:
        浄化後の画像（ImageNet正規化）
    """
    # [-1, 1]に変換
    x_diff = prepare_for_diffusion_from_norm(x_adv)
    batch_size = x_diff.shape[0]
    
    # ノイズ付加
    t_tensor = torch.full((batch_size,), t_start, device=device, dtype=torch.long)
    noise = torch.randn_like(x_diff, device=device)
    
    sqrt_alpha_cumprod_t = torch.sqrt(alphas_cumprod[t_start])
    sqrt_one_minus_alpha_cumprod_t = torch.sqrt(1.0 - alphas_cumprod[t_start])
    x_t = sqrt_alpha_cumprod_t * x_diff + sqrt_one_minus_alpha_cumprod_t * noise
    
    # 浄化（逆拡散）
    with torch.no_grad():
        for step in range(T_purify):
            current_t = t_start - step
            if current_t < 0:
                break
            
            t_batch = torch.full((batch_size,), current_t, device=device, dtype=torch.long)
            
            # ノイズ予測
            pred_noise = model(x_t, t_batch)
            
            # x0を再構成
            sqrt_alpha_cumprod = torch.sqrt(alphas_cumprod[current_t])
            sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - alphas_cumprod[current_t])
            x_0_pred = (x_t - sqrt_one_minus_alpha_cumprod * pred_noise) / sqrt_alpha_cumprod
            x_0_pred = torch.clamp(x_0_pred, -1.0, 1.0)
            
            # 次のステップへ
            if current_t > 0:
                prev_t = current_t - 1
                sqrt_alpha_cumprod_prev = torch.sqrt(alphas_cumprod[prev_t])
                sqrt_one_minus_alpha_cumprod_prev = torch.sqrt(1.0 - alphas_cumprod[prev_t])
                x_t = sqrt_alpha_cumprod_prev * x_0_pred + sqrt_one_minus_alpha_cumprod_prev * pred_noise
            else:
                x_t = x_0_pred
    
    # 最終的なx0を返す（ImageNet正規化に戻す）
    x_purified = prepare_from_diffusion_to_norm(x_t)
    return x_purified

# 浄化パラメータ
start_t = 80
T_purify = 50

print(f"DDPM浄化パラメータ:")
print(f"  start_t: {start_t}")
print(f"  T_purify: {T_purify}")
print(f"  Total timesteps: {T}")

DDPM浄化パラメータ:
  start_t: 80
  T_purify: 50
  Total timesteps: 1000


## 評価実行：AutoAttack + DDPM浄化

元画像で正解したサンプルのみを評価対象とします。

In [7]:
import os
import torch
import gc
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from torchvision.utils import save_image

# 結果保存用リスト
results = []

# 出力ディレクトリ
output_dir = "/mnt/data1/gotou/kaggle/ddpm/ddpm_auto/autoattack_results"
os.makedirs(output_dir, exist_ok=True)

# --- 正解サンプル抽出 ---
print("正解サンプルを抽出中...")
correct_samples = []
correct_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = (torch.sigmoid(outputs) > 0.5).float().squeeze()

        # 正解のみ抽出
        correct_mask = (preds == labels.float())
        if correct_mask.any():
            correct_samples.append(images[correct_mask])
            correct_labels.append(labels[correct_mask])

# 結合して先頭100枚のみ使用
all_correct_images = torch.cat(correct_samples, dim=0)[:100]
all_correct_labels = torch.cat(correct_labels, dim=0)[:100]

print(f"正解サンプル数（先頭100枚のみ）: {len(all_correct_images)}")

# --- AutoAttack 初期化 ---
adversary = AutoAttack(
    wrapped_model,
    norm='Linf',
    eps=epsilon,
    version='standard',
    verbose=False
)

# --- 評価ループ ---
batch_size_aa = 32
num_batches = (len(all_correct_images) + batch_size_aa - 1) // batch_size_aa

all_l2_norms_adv = []
all_linf_norms_adv = []
all_l2_norms_purified = []
all_linf_norms_purified = []
all_pred_orig = []
all_pred_adv = []
all_pred_purified = []

print("\nAutoAttack実行中...")

for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size_aa
    end_idx = min(start_idx + batch_size_aa, len(all_correct_images))

    images_batch = all_correct_images[start_idx:end_idx]
    labels_batch = all_correct_labels[start_idx:end_idx]

    # --- AutoAttack 実行 ---
    adv_images = adversary.run_standard_evaluation(
        images_batch, labels_batch.long(), bs=len(images_batch)
    )

    # --- DDPM 浄化 ---
    purified_images = diffusion_purify(adv_images, ddpm_model, start_t, T_purify, device)

    # --- 予測 ---
    with torch.no_grad():
        outputs_orig = model(images_batch)
        outputs_adv = model(adv_images)
        outputs_purified = model(purified_images)

        pred_orig = (torch.sigmoid(outputs_orig) > 0.5).float().squeeze()
        pred_adv = (torch.sigmoid(outputs_adv) > 0.5).float().squeeze()
        pred_purified = (torch.sigmoid(outputs_purified) > 0.5).float().squeeze()

    # --- L2 / L∞ノルム計算 ---
    diff_adv = (adv_images - images_batch).view(len(images_batch), -1)
    diff_purified = (purified_images - images_batch).view(len(images_batch), -1)

    l2_norm_adv = torch.norm(diff_adv, p=2, dim=1).cpu().numpy()
    linf_norm_adv = torch.norm(diff_adv, p=float('inf'), dim=1).cpu().numpy()
    l2_norm_purified = torch.norm(diff_purified, p=2, dim=1).cpu().numpy()
    linf_norm_purified = torch.norm(diff_purified, p=float('inf'), dim=1).cpu().numpy()

    # --- 結果保存 ---
    all_l2_norms_adv.extend(l2_norm_adv)
    all_linf_norms_adv.extend(linf_norm_adv)
    all_l2_norms_purified.extend(l2_norm_purified)
    all_linf_norms_purified.extend(linf_norm_purified)
    all_pred_orig.extend(pred_orig.cpu().numpy())
    all_pred_adv.extend(pred_adv.cpu().numpy())
    all_pred_purified.extend(pred_purified.cpu().numpy())

    for i in range(len(images_batch)):
        global_idx = start_idx + i
        results.append({
            'sample_idx': global_idx,
            'true_label': labels_batch[i].item(),
            'pred_original': pred_orig[i].item(),
            'pred_adversarial': pred_adv[i].item(),
            'pred_purified': pred_purified[i].item(),
            'l2_norm_adversarial': l2_norm_adv[i],
            'linf_norm_adversarial': linf_norm_adv[i],
            'l2_norm_purified': l2_norm_purified[i],
            'linf_norm_purified': linf_norm_purified[i],
            'attack_success': int(pred_adv[i].item() != labels_batch[i].item()),
            'purification_recovery': int(pred_purified[i].item() == labels_batch[i].item())
        })

        # --- 数枚だけ可視化用に保存 ---
        if batch_idx < 3 and i < 3:  # 最初の3バッチ×3サンプル
            triplet = torch.cat([images_batch[i], adv_images[i], purified_images[i]], dim=-1)
            save_path = os.path.join(output_dir, f"sample_{global_idx:03d}.png")
            save_image(triplet, save_path)

    print(f"  Batch {batch_idx+1}/{num_batches} 完了")

    # --- メモリ開放 ---
    del adv_images, purified_images, outputs_orig, outputs_adv, outputs_purified
    torch.cuda.empty_cache()
    gc.collect()

print("\n評価完了")
print(f"総サンプル数: {len(results)}")


正解サンプルを抽出中...


KeyboardInterrupt: 

In [ ]:
# 結果の集計とCSV/TXT出力
import pandas as pd

# DataFrameに変換
df_results = pd.DataFrame(results)

# CSV保存
csv_path = os.path.join(output_dir, "autoattack_ddpm_detailed.csv")
df_results.to_csv(csv_path, index=False)
print(f"詳細結果をCSVに保存: {csv_path}")

# サマリー統計
labels = all_correct_labels.cpu().numpy()
pred_orig = np.array(all_pred_orig)
pred_adv = np.array(all_pred_adv)
pred_purified = np.array(all_pred_purified)

# 精度計算
acc_orig = (pred_orig == labels).mean()
acc_adv = (pred_adv == labels).mean()
acc_purified = (pred_purified == labels).mean()

# 攻撃成功率
attack_success_rate = (pred_adv != labels).mean()

# 浄化による回復率
recovery_rate = ((pred_adv != labels) & (pred_purified == labels)).sum() / (pred_adv != labels).sum() if (pred_adv != labels).sum() > 0 else 0

# L2/L∞ノルム統計
l2_adv_mean = np.mean(all_l2_norms_adv)
l2_adv_std = np.std(all_l2_norms_adv)
linf_adv_mean = np.mean(all_linf_norms_adv)
linf_adv_std = np.std(all_linf_norms_adv)

l2_purified_mean = np.mean(all_l2_norms_purified)
l2_purified_std = np.std(all_l2_norms_purified)
linf_purified_mean = np.mean(all_linf_norms_purified)
linf_purified_std = np.std(all_linf_norms_purified)

# TXT保存
txt_path = os.path.join(output_dir, "autoattack_ddpm_summary.txt")
with open(txt_path, 'w') as f:
    f.write("=== AutoAttack + DDPM Purification 評価結果 ===\n\n")
    f.write("【攻撃パラメータ】\n")
    f.write(f"  攻撃手法: AutoAttack (APGD-CE, APGD-DLR, FAB, Square)\n")
    f.write(f"  Epsilon: {epsilon:.4f}\n")
    f.write(f"  Norm: L∞\n\n")
    f.write("【浄化パラメータ】\n")
    f.write(f"  手法: DDPM x0 reconstruction\n")
    f.write(f"  start_t: {start_t}\n")
    f.write(f"  T_purify: {T_purify}\n\n")
    f.write("【精度】\n")
    f.write(f"  元画像: {acc_orig*100:.2f}% (全サンプルで正解)\n")
    f.write(f"  敵対的サンプル: {acc_adv*100:.2f}%\n")
    f.write(f"  浄化後: {acc_purified*100:.2f}%\n\n")
    f.write("【攻撃・防御統計】\n")
    f.write(f"  攻撃成功率: {attack_success_rate*100:.2f}%\n")
    f.write(f"  浄化による回復率: {recovery_rate*100:.2f}%\n\n")
    f.write("【L2ノルム（敵対的摂動）】\n")
    f.write(f"  平均: {l2_adv_mean:.4f}\n")
    f.write(f"  標準偏差: {l2_adv_std:.4f}\n\n")
    f.write("【L∞ノルム（敵対的摂動）】\n")
    f.write(f"  平均: {linf_adv_mean:.4f}\n")
    f.write(f"  標準偏差: {linf_adv_std:.4f}\n\n")
    f.write("【L2ノルム（浄化後の差分）】\n")
    f.write(f"  平均: {l2_purified_mean:.4f}\n")
    f.write(f"  標準偏差: {l2_purified_std:.4f}\n\n")
    f.write("【L∞ノルム（浄化後の差分）】\n")
    f.write(f"  平均: {linf_purified_mean:.4f}\n")
    f.write(f"  標準偏差: {linf_purified_std:.4f}\n\n")
    f.write(f"総評価サンプル数: {len(results)}\n")

print(f"サマリー統計をTXTに保存: {txt_path}")

# 混同行列の計算
cm_orig = confusion_matrix(labels, pred_orig)
cm_adv = confusion_matrix(labels, pred_adv)
cm_purified = confusion_matrix(labels, pred_purified)

print("\n=== サマリー統計 ===")
print(f"元画像精度: {acc_orig*100:.2f}%")
print(f"敵対的サンプル精度: {acc_adv*100:.2f}%")
print(f"浄化後精度: {acc_purified*100:.2f}%")
print(f"攻撃成功率: {attack_success_rate*100:.2f}%")
print(f"浄化回復率: {recovery_rate*100:.2f}%")

In [ ]:
# 混同行列の可視化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 元画像
sns.heatmap(cm_orig, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title(f'元画像の混同行列\n精度: {acc_orig*100:.2f}%', fontsize=14)
axes[0].set_xlabel('予測ラベル', fontsize=12)
axes[0].set_ylabel('真のラベル', fontsize=12)

# 敵対的サンプル
sns.heatmap(cm_adv, annot=True, fmt='d', cmap='Reds', ax=axes[1], cbar=False)
axes[1].set_title(f'敵対的サンプルの混同行列\n精度: {acc_adv*100:.2f}%', fontsize=14)
axes[1].set_xlabel('予測ラベル', fontsize=12)
axes[1].set_ylabel('真のラベル', fontsize=12)

# 浄化後
sns.heatmap(cm_purified, annot=True, fmt='d', cmap='Greens', ax=axes[2], cbar=False)
axes[2].set_title(f'DDPM浄化後の混同行列\n精度: {acc_purified*100:.2f}%', fontsize=14)
axes[2].set_xlabel('予測ラベル', fontsize=12)
axes[2].set_ylabel('真のラベル', fontsize=12)

plt.tight_layout()
cm_path = os.path.join(output_dir, "confusion_matrices.png")
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"混同行列を保存: {cm_path}")

In [ ]:
# ノルム分布のヒストグラム
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# L2ノルム（敵対的摂動）
axes[0, 0].hist(all_l2_norms_adv, bins=50, color='crimson', alpha=0.7, edgecolor='black')
axes[0, 0].set_title(f'L2ノルム分布（敵対的摂動）\n平均: {l2_adv_mean:.4f}, 標準偏差: {l2_adv_std:.4f}', fontsize=12)
axes[0, 0].set_xlabel('L2ノルム', fontsize=10)
axes[0, 0].set_ylabel('頻度', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# L∞ノルム（敵対的摂動）
axes[0, 1].hist(all_linf_norms_adv, bins=50, color='darkred', alpha=0.7, edgecolor='black')
axes[0, 1].set_title(f'L∞ノルム分布（敵対的摂動）\n平均: {linf_adv_mean:.4f}, 標準偏差: {linf_adv_std:.4f}', fontsize=12)
axes[0, 1].set_xlabel('L∞ノルム', fontsize=10)
axes[0, 1].set_ylabel('頻度', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# L2ノルム（浄化後の差分）
axes[1, 0].hist(all_l2_norms_purified, bins=50, color='forestgreen', alpha=0.7, edgecolor='black')
axes[1, 0].set_title(f'L2ノルム分布（浄化後の差分）\n平均: {l2_purified_mean:.4f}, 標準偏差: {l2_purified_std:.4f}', fontsize=12)
axes[1, 0].set_xlabel('L2ノルム', fontsize=10)
axes[1, 0].set_ylabel('頻度', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# L∞ノルム（浄化後の差分）
axes[1, 1].hist(all_linf_norms_purified, bins=50, color='darkgreen', alpha=0.7, edgecolor='black')
axes[1, 1].set_title(f'L∞ノルム分布（浄化後の差分）\n平均: {linf_purified_mean:.4f}, 標準偏差: {linf_purified_std:.4f}', fontsize=12)
axes[1, 1].set_xlabel('L∞ノルム', fontsize=10)
axes[1, 1].set_ylabel('頻度', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
hist_path = os.path.join(output_dir, "norm_histograms.png")
plt.savefig(hist_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"ノルムヒストグラムを保存: {hist_path}")

In [ ]:
# ノルムのボックスプロット
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# L2ノルム比較
data_l2 = [all_l2_norms_adv, all_l2_norms_purified]
bp1 = axes[0].boxplot(data_l2, labels=['敵対的摂動', '浄化後の差分'], patch_artist=True)
bp1['boxes'][0].set_facecolor('crimson')
bp1['boxes'][1].set_facecolor('forestgreen')
axes[0].set_title('L2ノルムの比較', fontsize=14)
axes[0].set_ylabel('L2ノルム', fontsize=12)
axes[0].grid(True, alpha=0.3)

# L∞ノルム比較
data_linf = [all_linf_norms_adv, all_linf_norms_purified]
bp2 = axes[1].boxplot(data_linf, labels=['敵対的摂動', '浄化後の差分'], patch_artist=True)
bp2['boxes'][0].set_facecolor('darkred')
bp2['boxes'][1].set_facecolor('darkgreen')
axes[1].set_title('L∞ノルムの比較', fontsize=14)
axes[1].set_ylabel('L∞ノルム', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
box_path = os.path.join(output_dir, "norm_boxplots.png")
plt.savefig(box_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"ノルムボックスプロットを保存: {box_path}")

In [ ]:
# 攻撃成功/失敗とノルムの関係（散布図）
attack_success = df_results['attack_success'].values
purif_recovery = df_results['purification_recovery'].values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# L2ノルム vs 攻撃成功
colors_success = ['green' if s == 0 else 'red' for s in attack_success]
axes[0].scatter(all_l2_norms_adv, all_linf_norms_adv, c=colors_success, alpha=0.6, s=20)
axes[0].set_xlabel('L2ノルム（敵対的摂動）', fontsize=12)
axes[0].set_ylabel('L∞ノルム（敵対的摂動）', fontsize=12)
axes[0].set_title('攻撃成功/失敗とノルムの関係\n（緑: 失敗, 赤: 成功）', fontsize=14)
axes[0].grid(True, alpha=0.3)

# L2ノルム vs 浄化回復
# 攻撃成功したサンプルのみを対象
attacked_indices = np.where(attack_success == 1)[0]
l2_attacked = np.array(all_l2_norms_purified)[attacked_indices]
linf_attacked = np.array(all_linf_norms_purified)[attacked_indices]
recovery_attacked = purif_recovery[attacked_indices]

colors_recovery = ['blue' if r == 1 else 'orange' for r in recovery_attacked]
axes[1].scatter(l2_attacked, linf_attacked, c=colors_recovery, alpha=0.6, s=20)
axes[1].set_xlabel('L2ノルム（浄化後の差分）', fontsize=12)
axes[1].set_ylabel('L∞ノルム（浄化後の差分）', fontsize=12)
axes[1].set_title('浄化による回復とノルムの関係（攻撃成功サンプルのみ）\n（青: 回復, オレンジ: 未回復）', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
scatter_path = os.path.join(output_dir, "norm_scatter_plots.png")
plt.savefig(scatter_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"散布図を保存: {scatter_path}")

In [ ]:
# サンプル画像の可視化（最初の8サンプル）
num_vis_samples = min(8, len(all_correct_images))

fig, axes = plt.subplots(num_vis_samples, 4, figsize=(16, num_vis_samples * 4))

# 正規化を戻す関数
def denormalize(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img_tensor * std + mean
    return torch.clamp(img, 0, 1)

for i in range(num_vis_samples):
    # 元画像
    img_orig = denormalize(all_correct_images[i].cpu())
    axes[i, 0].imshow(img_orig.permute(1, 2, 0).numpy())
    axes[i, 0].set_title(f'元画像 (ラベル: {int(all_correct_labels[i])})', fontsize=10)
    axes[i, 0].axis('off')
    
    # 敵対的サンプル
    img_adv = denormalize(torch.cat(all_adv_images, dim=0)[i])
    axes[i, 1].imshow(img_adv.permute(1, 2, 0).numpy())
    pred_text = f"予測: {int(all_pred_adv[i])}"
    axes[i, 1].set_title(f'AutoAttack\n{pred_text}', fontsize=10)
    axes[i, 1].axis('off')
    
    # 浄化後
    img_purified = denormalize(torch.cat(all_purified_images, dim=0)[i])
    axes[i, 2].imshow(img_purified.permute(1, 2, 0).numpy())
    pred_text = f"予測: {int(all_pred_purified[i])}"
    axes[i, 2].set_title(f'DDPM浄化後\n{pred_text}', fontsize=10)
    axes[i, 2].axis('off')
    
    # 差分（敵対的摂動）
    diff = (img_adv - img_orig).abs()
    axes[i, 3].imshow(diff.permute(1, 2, 0).numpy())
    axes[i, 3].set_title(f'摂動 (L2: {all_l2_norms_adv[i]:.2f})', fontsize=10)
    axes[i, 3].axis('off')

plt.tight_layout()
samples_path = os.path.join(output_dir, "sample_images.png")
plt.savefig(samples_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"サンプル画像を保存: {samples_path}")

print("\n" + "="*60)
print("すべての評価と可視化が完了しました！")
print("="*60)
print(f"\n出力ディレクトリ: {output_dir}")
print("生成されたファイル:")
print(f"  1. {csv_path}")
print(f"  2. {txt_path}")
print(f"  3. {cm_path}")
print(f"  4. {hist_path}")
print(f"  5. {box_path}")
print(f"  6. {scatter_path}")
print(f"  7. {samples_path}")